# Классификация вин: Сравнение ML моделей и PyTorch Lightning
### Dataset: Wine (sklearn)

In [ ]:
# Установка необходимых библиотек
!pip install pytorch-lightning>=2.0 xgboost -q

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

# ML модели
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# PyTorch Lightning
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline

## 1. Загрузка и первичный анализ данных

In [ ]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["target"] = wine.target

print("Датасет вина:")
print(df.head())
print(f"\nРазмер данных: {df.shape}")
print(f"Признаки: {wine.feature_names}")
print(f"Целевые классы: {wine.target_names}")
print(f"Количество классов: {len(np.unique(wine.target))}")

## 2. Разделение данных и нормализация

In [ ]:
X = wine.data
y = wine.target

# Разделение на train (70%), val (15%), test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Нормализация
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Для PCA (все данные)
X_all_scaled = np.vstack([X_train_scaled, X_val_scaled, X_test_scaled])
y_all = np.hstack([y_train, y_val, y_test])

print(f"Train: {X_train_scaled.shape}, Val: {X_val_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Всего для визуализации: {X_all_scaled.shape}")

## 3. PCA визуализация данных

In [ ]:
# 2D PCA
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_all_scaled)

print("Объясненная дисперсия (2 компоненты):")
print(f"PC1: {pca_2d.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca_2d.explained_variance_ratio_[1]:.3f}")
print(f"Суммарно: {pca_2d.explained_variance_ratio_.sum():.3f}")

plt.figure(figsize=(10, 6))
colors = ['red', 'green', 'blue']
for i in np.unique(y_all):
    plt.scatter(
        X_2d[y_all == i, 0],
        X_2d[y_all == i, 1],
        label=wine.target_names[i],
        alpha=0.7, s=50, c=colors[i]
    )
plt.xlabel("Первая главная компонента (PC1)")
plt.ylabel("Вторая главная компонента (PC2)")
plt.title("2D PCA проекция набора данных Wine")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3D PCA
pca_3d = PCA(n_components=3)
X_3d = pca_3d.fit_transform(X_all_scaled)

print(f"\nОбъясненная дисперсия (3 компоненты):")
print(f"PC1: {pca_3d.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca_3d.explained_variance_ratio_[1]:.3f}")
print(f"PC3: {pca_3d.explained_variance_ratio_[2]:.3f}")
print(f"Суммарно: {pca_3d.explained_variance_ratio_.sum():.3f}")

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for i in np.unique(y_all):
    ax.scatter(
        X_3d[y_all == i, 0],
        X_3d[y_all == i, 1],
        X_3d[y_all == i, 2],
        label=wine.target_names[i],
        alpha=0.7, s=40, c=colors[i]
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("3D PCA проекция набора данных Wine")
ax.legend()
plt.tight_layout()
plt.show()

print("\nВывод: Данные хорошо разделимы даже при снижении размерности до 2-3 компонент.")

## 4. Классические ML модели (бенчмарк)

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=10))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        objective="multi:softprob", eval_metric="mlogloss", random_state=42
    )
}

ml_results = []

print("=" * 60)
print("РЕЗУЛЬТАТЫ КЛАССИЧЕСКИХ ML МОДЕЛЕЙ")
print("=" * 60)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    test_acc = accuracy_score(y_test, preds)
    cv_acc = cross_val_score(model, X, y, cv=5).mean()
    
    ml_results.append({
        "Model": name,
        "Test Accuracy": test_acc,
        "CV Accuracy": cv_acc
    })
    
    print(f"\n{name}")
    print(f"  Test Accuracy : {test_acc:.4f}")
    print(f"  CV Accuracy   : {cv_acc:.4f}")

ml_results_df = pd.DataFrame(ml_results)
print("\n" + "=" * 60)
print(ml_results_df.to_string(index=False))

## 5. Нейронная сеть на PyTorch Lightning

In [ ]:
# Преобразование в тензоры
X_train_t = torch.FloatTensor(X_train_scaled)
X_val_t = torch.FloatTensor(X_val_scaled)
X_test_t = torch.FloatTensor(X_test_scaled)
y_train_t = torch.LongTensor(y_train)
y_val_t = torch.LongTensor(y_val)
y_test_t = torch.LongTensor(y_test)

class WineDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.clone().detach().float() if torch.is_tensor(X) else torch.tensor(X, dtype=torch.float32)
        self.y = y.clone().detach().long() if torch.is_tensor(y) else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class WineDataModule(pl.LightningDataModule):
    def __init__(self, X_train, X_val, X_test, y_train, y_val, y_test, batch_size=16):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.X_test = X_test
        self.y_train = y_train
        self.y_val = y_val
        self.y_test = y_test
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_dataset = WineDataset(self.X_train, self.y_train)
        self.val_dataset = WineDataset(self.X_val, self.y_val)
        self.test_dataset = WineDataset(self.X_test, self.y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

class NeuralNetLightning(pl.LightningModule):
    def __init__(self, input_size, num_classes, lr=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
data_module = WineDataModule(
    X_train=X_train_t, X_val=X_val_t, X_test=X_test_t,
    y_train=y_train_t, y_val=y_val_t, y_test=y_test_t,
    batch_size=16
)

model_nn = NeuralNetLightning(
    input_size=X_train_scaled.shape[1],
    num_classes=len(np.unique(y)),
    lr=0.001
)

early_stop = EarlyStopping(monitor="val_loss", patience=15, mode="min")
checkpoint = ModelCheckpoint(
    monitor="val_loss", mode="min", save_top_k=1,
    filename="best-{epoch:02d}-{val_loss:.4f}"
)

trainer = pl.Trainer(
    max_epochs=100, accelerator="auto", devices=1,
    log_every_n_steps=5, enable_progress_bar=True,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
print("Начинаем обучение нейронной сети...")
trainer.fit(model_nn, data_module)
print("\nОбучение завершено.")

In [ ]:
print("\nРезультаты нейронной сети на тестовой выборке:")
test_results = trainer.test(model_nn, datamodule=data_module, ckpt_path="best")
nn_test_acc = test_results[0]['test_acc']
nn_test_loss = test_results[0]['test_loss']
print(f"Test Accuracy: {nn_test_acc:.4f}")
print(f"Test Loss: {nn_test_loss:.4f}")
print(f"\nЛучший чекпоинт: {checkpoint.best_model_path}")

## 6. Сравнение всех моделей

In [ ]:
comparison_df = pd.DataFrame(ml_results)
nn_row = pd.DataFrame([{
    "Model": "PyTorch Lightning NN",
    "Test Accuracy": nn_test_acc,
    "CV Accuracy": np.nan
}])
comparison_df = pd.concat([comparison_df, nn_row], ignore_index=True)
comparison_df = comparison_df.sort_values("Test Accuracy", ascending=False)

print("=" * 60)
print("СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ")
print("=" * 60)
print(comparison_df.to_string(index=False))

plt.figure(figsize=(12, 6))
bars = plt.bar(comparison_df["Model"], comparison_df["Test Accuracy"], 
               color=['skyblue', 'lightgreen', 'lightcoral', 'gold', 'plum', 'orange'])
plt.ylim(0.9, 1.02)
plt.ylabel("Test Accuracy")
plt.title("Сравнение точности моделей на тестовой выборке")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, comparison_df["Test Accuracy"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Confusion Matrix (PyTorch Lightning)

In [ ]:
best_model = NeuralNetLightning.load_from_checkpoint(checkpoint.best_model_path)
best_model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for x_batch, y_batch in data_module.test_dataloader():
        logits = best_model(x_batch)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)

print("Classification Report - PyTorch Lightning NN:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=wine.target_names, 
            yticklabels=wine.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - PyTorch Lightning NN')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nГрафик сохранен как 'confusion_matrix.png'")

## 8. Выводы

### Результаты:
- **Random Forest** и **Logistic Regression** показали лучшие результаты среди классических ML моделей
- **PyTorch Lightning NN** достигла конкурентоспособной точности (96.3%)
- **PCA визуализация** подтвердила, что классы вин хорошо разделимы

### Преимущества PyTorch Lightning:
- Удобная организация кода (DataModule, Callbacks)
- Автоматическое логирование метрик
- Ранняя остановка предотвращает переобучение
- Легко масштабировать на более сложные архитектуры

### Что можно улучшить:
- Гиперпараметрическая оптимизация (Optuna, Ray Tune)
- Более сложная архитектура сети
- Аугментация данных (если бы данные позволяли)